# Lesson 04 - ഉപകരണ ഉപയോഗ രൂപകൽപ്പന മാതൃക

ഈ പാഠത്തിൽ നിങ്ങൾ Microsoft Agent Framework (Python) ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കായി **ഉപകരണ ഉപയോഗം** രൂപകൽപ്പന മാതൃക പഠിക്കും. നാം ഉൾക്കൊള്ളുന്നു:

- `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച് ഫംഗ്ഷൻ ഉപകരണങ്ങൾ നിർവചിക്കുക, ടൈപ്പുചെയ്ത പാരാമീറ്ററുകളുമായി
- ഓരോ ഉപകരണവും എന്ത് ചെയ്യുന്നതെന്ന് മോഡലിന് അറിയാൻ ഉപകരണ സ്കീമകൾ നൽകുക
- `approval_mode` ഉപയോഗിച്ച് ഉപകരണം പ്രവർത്തനം നിയന്ത്രിക്കുക
- Pydantic മോഡലുകളും `response_format` ഉപയോഗിച്ചുള്ള **സംഘടിത output** നൽകുക

പ്രസംഗം ഒരു **സഞ്ചാര ബുക്കിംഗ് ഏജന്റാണ്** അത് ഗണ്യസ്ഥാനങ്ങൾ തിരയുകയിലും, ലഭ്യത പരിശോധിക്കുകയിലും, ഫ്ലൈറ്റ് വിവരങ്ങൾ നേടുകയുമാണ്.


## ക്രമീകരണം


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ഡെക്കറേറ്ററോടെയാണ് ഉപകരണങ്ങൾ നിർവചിക്കുന്നത്

`@tool` ഡെക്കറേറ്റർ ഒരു സാധാരണ Python ഫംക്ഷൻ ഒരു ഏജന്റ് വിളിക്കാൻ കഴിയുന്ന ഉപകരണമായി മാറുന്നു.
പ്രധാനപ്പെട്ട കാര്യങ്ങൾ:

- **ഡോക്സ്ട്രിംഗ്** മോഡൽ കാണുന്ന ഉപകരണ വിവരണമായിരിക്കും.
- **টাইപ്പ് അനോട്ടേഷനുകൾ** (വിവരണങ്ങളോടുള്ള `Annotated` ഉൾപ്പെടെ) ഉപകരണ സ്കീമ നിർവചിക്കുന്നു.
- `approval_mode` ഓരോ വിളിയും നടപ്പിലാക്കുന്നതിന് മുമ്പ് ഉപഭോക്താവ് അംഗീകരിക്കണമോ എന്ന് നിയന്ത്രിക്കുന്നു.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## ഒന്നിലധികം ടൂളുകൾ ഉപയോഗിച്ച് ഏജന്റ് സൃഷ്ടിക്കൽ

ഉപയോക്താവിന്റെ ചോദ്യത്തിന് മറുപടി നൽകാൻ മോഡൽ ആവശ്യപ്പെടുന്ന ഏത് ടൂൾസ് ആയാലും ലഭ്യമാക്കാൻ മൂന്നു ടൂളുകളും ക്ലയന്റിന് നൽകുക.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ടൂളുകൾ ഉപയോഗിച്ച് ഘടിതമായ ഔട്ട്പുട്ട്

`response_format` ഒരു പൈഡാന്റിക് മോഡലായി ക്രമീകരിക്കുന്നത്, ഏജന്റ് സ്വതന്ത്രമായ ടെക്സ്റ്റിന്റെ പകരം ഒരു നന്നായി ടൈപ്പ് ചെയ്ത JSON ഒബ്ജക്റ്റ് മടക്കി നൽകാൻ നിർബന്ധിക്കുന്നു. ഇത് ഡൗൺസ്ട്രീം കോഡ് ഫലത്തെ പ്രോഗ്രാമാറ്റിക് ആയി ഉപയോഗിക്കേണ്ട സമയത്തു പ്രയോജനപ്പെടുന്നു.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ടൂൾ അംഗീകാര പാറ്റേണുകൾ

`@tool` ൽ `approval_mode` പാരാമീറ്റർ ടൂൾ കോൾകൾ പ്രവർത്തിപ്പിക്കുന്നതിന് മുമ്പ് മനുഷ്യ അംഗീകാരം ആവശ്യമാണ് എന്നെക്കുറിച്ച് നിയന്ത്രിക്കുന്നു:

| മോഡ് | പെരുമാറ്റം |
|---|---|
| `"never_require"` | ടൂൾ സ്വയം പ്രവർത്തിക്കും — ഉപയോക്തൃ സ്ഥിരീകരണം ആവശ്യമില്ല. |
| `"always_require"` | ഓരോ കോൾക്കും മനുഷ്യന്റെ അംഗീകാരമുണ്ടായിരിക്കണം പ്രവർത്തിപ്പിക്കുന്നതിന് മുമ്പ്. |

ഫ്ലൈറ്റ് ബുക്ക് ചെയ്യുക, ക്രെഡിറ്റ് കാർഡ് ചാർജ് ചെയ്യുക പോലുള്ള സൈഡ്-എഫക്റ്റുകൾ ഉള്ള ടൂളുകൾക്കായി `"always_require"` ഉപയോഗിക്കുക, അതിനാൽ മനുഷ്യൻ പ്രക്രിയയിൽ തുടങ്ങിയിരിക്കുന്നു.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## സാരാംശം

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിച്ചത്:

1. ടൂൾ സ്കീമയായി പ്രവർത്തിക്കുന്ന ടൈപ്പുചെയ്ത പാരാമീറ്ററുകളോടുകൂടിയ ഡോക്സ്ട്രിംഗുകളുള്ള `@tool` ഡെക്കറേറ്റർ ഉപയോഗിച്ച് **ടൂളുകൾ നിർവചിക്കുന്നത്**.
2. ഏജന്റ് സങ്കീർണമായ ചോദ്യങ്ങൾക്ക് ഉത്തരം നൽകാൻ തിരിച്ചടിയായി വിളിക്കാൻ കഴിയുന്നവിധം **പല ടൂളുകൾ സംയോജിപ്പിക്കൽ**.
3. Pydantic മോഡൽ `response_format` ആയി നൽകുക വഴി **സംരಚಿತമായ ഔട്ട്പുട്ട് മടങ്ങുക**.
4. സങ്കീർണമായ പ്രവർത്തനങ്ങൾക്കായി മനുഷ്യനെ ഇടപെടുന്ന നിലയിൽ സൂക്ഷിക്കാൻ `approval_mode` ഉപയോഗിച്ച് **ടൂൾ അംഗീകാര നിയന്ത്രണം നിലനിർത്തുക**.

വൈശാല്യത്തോടെ, പ്രൊഡക്ഷൻ-സജ്ജമായ ഏജന്റുകളെ നിർമ്മിക്കുന്നതിന് ഈ മാതൃകകൾ നിബന്ധനകളും, സുരക്ഷിതമായി ബാഹ്യ സംവിധാനവുമായ സംവദിക്കാൻ കഴിയുന്നതുമായ എജന്റുകളെ ആധാരമാക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**പരാമർശന കുറിപ്പ്**:  
ഈ ഡോക്യുമെന്റ് AI തർജ്ജമ സേവനമായ [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് തർജ്ജമ ചെയ്തിരിക്കുന്നു. ഞങ്ങൾ തർക്കരഹിതമായതിന് പരിശ്രമിക്കുന്നുവെങ്കിലും, സ്വയം പ്രവർത്തിക്കുന്ന തർജ്ജമയിൽ പിഴവുകളോ അശുദ്ധികളോ ഉണ്ടാകാൻ സാധ്യതയുണ്ടെന്ന് ദയവായി ശ്രദ്ധിക്കണം. സർക്കാരികമായ പ്രമാണം അതിന്റെ മാതൃഭാഷയിലുള്ള യഥാർത്ഥ ഡോക്യുമെന്റ് ആണ് സർവകാരിയായ ഉദ്ദേശ്യത്തിന്റെ ഉറവിടം. അതിവിശേഷ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മാനവ തർജ്ജമ ശിപാർശ ചെയ്യപ്പെടുന്നു. ഈ തർജ്മ ഉപയോഗിച്ചുള്ള തെറ്റിദ്ധാരണകൾക്കും തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദിത്വം വഹിക്കുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
